# AILA 2019 Supervised BiLSTM Legal Query-Document Relevance Model

This notebook implements a **Siamese-style BiLSTM relevance model** trained on the **AILA 2019 dataset** for 50 epochs using PyTorch.

### Model Architecture Overview:
- **Dual Encoder**: Query & Document tokens $\to$ Embedding $\to$ BiLSTM $\to$ Attention Pooling $\to$ $u$ (query vector), $v$ (document vector)
- **Matching Layer**: Computes matching features:
  1. $u$ (`query_representation`)
  2. $v$ (`document_representation`)
  3. $|u - v|$ (`absolute_difference`)
  4. $u \odot v$ (`elementwise_product`)
- **MLP Classifier**: Concatenates $[u, v, |u - v|, u \odot v]$ $\to$ Linear $\to$ ReLU $\to$ Dropout $\to$ Linear Logit $\to$ `BCEWithLogitsLoss`
- **Ranking Evaluation**: P@5, Recall@5, MRR, MAP, NDCG@5, NDCG@10

In [ ]:
# Cell 1: Environment Setup & Compute Device Detection
import os
import sys
import site
import random

# Ensure project root is in sys.path when running from notebooks/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

user_site = site.getusersitepackages()
if user_site and user_site not in sys.path:
    sys.path.insert(0, user_site)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Auto-detect GPU / CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Device] Active Compute Device: {device}')
if torch.cuda.is_available():
    print(f'[Device] GPU Name: {torch.cuda.get_device_name(0)}')

# Reproducibility seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
# Cell 2: Hyperparameter Configuration
CONFIG = {
    'BATCH_SIZE': 32,
    'LEARNING_RATE': 0.001,
    'EPOCHS': 50,
    'EMBEDDING_DIM': 128,
    'HIDDEN_DIM': 128,
    'NUM_LAYERS': 1,
    'DROPOUT': 0.2,
    'MAX_QUERY_LENGTH': 256,
    'MAX_DOCUMENT_LENGTH': 512,
    'NUM_NEGATIVES': 5,
    'EARLY_STOPPING_PATIENCE': 10
}
print('[Config] Hyperparameters:', CONFIG)

In [ ]:
# Cell 3: Data Loading & Query-Level Split (Train Q1-Q8, Val Q9-Q10, Test Q11-Q50)
from src.data_loader import AILADataLoader

data_dir = os.path.join(project_root, 'data', 'raw', 'AILA2019')
loader = AILADataLoader(data_dir)

queries = loader.load_queries()
casedocs = loader.load_case_docs()
qrels = loader.load_qrels_priorcases()

train_qids, val_qids, test_qids = loader.get_query_splits(queries, qrels)
print(f'[Data Split] Train Queries ({len(train_qids)}):', train_qids)
print(f'[Data Split] Val Queries   ({len(val_qids)}):', val_qids)
print(f'[Data Split] Test Queries  ({len(test_qids)}):', len(test_qids), 'queries (Q11-Q50)')

In [ ]:
# Cell 4: Vocabulary Fitting (Strictly on Train Split to Prevent Leakage)
from src.preprocessing import Vocabulary

train_texts = [queries[q] for q in train_qids if q in queries]
for qid in train_qids:
    for doc_id in qrels.get(qid, set()):
        if doc_id in casedocs:
            train_texts.append(casedocs[doc_id])

vocab = Vocabulary()
vocab.build_vocab(train_texts, max_vocab_size=30000, min_freq=2)
print(f'[Vocab] Vocabulary built on train split. Size: {len(vocab)} tokens.')

In [ ]:
# Cell 5: PyTorch Binary Pair Dataset & DataLoader
from src.dataset import BiLSTMPairDataset, collate_bilstm_pairs

train_dataset = BiLSTMPairDataset(
    qids=train_qids,
    queries=queries,
    casedocs=casedocs,
    qrels=qrels,
    vocab=vocab,
    num_negatives=CONFIG['NUM_NEGATIVES'],
    max_q_len=CONFIG['MAX_QUERY_LENGTH'],
    max_d_len=CONFIG['MAX_DOCUMENT_LENGTH']
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=True,
    collate_fn=lambda b: collate_bilstm_pairs(b, max_q_len=CONFIG['MAX_QUERY_LENGTH'], max_d_len=CONFIG['MAX_DOCUMENT_LENGTH'])
)
print(f'[Dataset] Number of training batches: {len(train_loader)}')

In [ ]:
# Cell 6: Model & Optimizer Initialization
from src.models.bilstm import SiameseBiLSTMRelevanceModel
from src.loss import get_relevance_loss

model = SiameseBiLSTMRelevanceModel(
    vocab_size=len(vocab),
    embedding_dim=CONFIG['EMBEDDING_DIM'],
    hidden_dim=CONFIG['HIDDEN_DIM'],
    num_layers=CONFIG['NUM_LAYERS'],
    dropout=CONFIG['DROPOUT']
).to(device)

criterion = get_relevance_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['LEARNING_RATE'])
print(model)

In [ ]:
# Cell 7: SANITY CHECK (Before Full Training)
sanity_batch = next(iter(train_loader))
s_q_ids = sanity_batch['query_ids'].to(device)
s_d_ids = sanity_batch['doc_ids'].to(device)
s_labels = sanity_batch['labels'].to(device)

model.eval()
with torch.no_grad():
    s_logits = model(s_q_ids, s_d_ids)

print('=' * 50)
print('              SANITY CHECK OUTPUT')
print('=' * 50)
print(f'query tensor shape:    {s_q_ids.shape}')
print(f'document tensor shape: {s_d_ids.shape}')
print(f'label tensor shape:    {s_labels.shape}')
print(f'model output shape:    {s_logits.shape}')

# Backward pass sanity check
model.train()
optimizer.zero_grad()
s_out = model(s_q_ids, s_d_ids)
s_loss = criterion(s_out, s_labels)
s_loss.backward()
optimizer.step()
print(f'Sanity Batch Loss:    {s_loss.item():.4f}')
print('Sanity check passed successfully!')

In [ ]:
# Cell 8: 50-Epoch Full Training Loop with Early Stopping & Checkpointing
from src.dataset import pad_sequence
from src.preprocessing import TFIDFRetriever
from src.evaluation import calculate_metrics

best_val_map = 0.0
best_epoch = 0
patience_counter = 0
history = []
checkpoint_dir = os.path.join(project_root, 'models', 'bilstm')
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'bilstm_model.pt')

# Pre-compute candidate pools for fast validation scoring
tfidf_retriever = TFIDFRetriever(casedocs)
bm25_candidate_pools = {}
for qid in queries:
    ranked_docs = tfidf_retriever.rank_documents(queries[qid])
    top_docs = [doc_id for doc_id, _ in ranked_docs]
    pos_docs = list(qrels.get(qid, set()))
    combined = list(dict.fromkeys(pos_docs + top_docs[:200]))
    bm25_candidate_pools[qid] = combined

def evaluate_model(eval_qids, candidate_pool_size=150):
    model.eval()
    rankings = {}
    with torch.no_grad():
        for qid in eval_qids:
            q_text = queries[qid]
            q_padded = pad_sequence(vocab.text_to_ids(q_text, max_len=CONFIG['MAX_QUERY_LENGTH']), CONFIG['MAX_QUERY_LENGTH'])
            q_ids = torch.tensor([q_padded], dtype=torch.long, device=device)
            
            if candidate_pool_size is not None and qid in bm25_candidate_pools:
                target_doc_ids = bm25_candidate_pools[qid][:candidate_pool_size]
            else:
                target_doc_ids = list(casedocs.keys())
                
            scores = []
            chunk_size = 128
            for i in range(0, len(target_doc_ids), chunk_size):
                chunk_doc_ids = target_doc_ids[i:i+chunk_size]
                doc_padded_list = [
                    pad_sequence(vocab.text_to_ids(casedocs[doc_id], max_len=CONFIG['MAX_DOCUMENT_LENGTH']), CONFIG['MAX_DOCUMENT_LENGTH'])
                    for doc_id in chunk_doc_ids
                ]
                doc_tensor = torch.tensor(doc_padded_list, dtype=torch.long, device=device)
                q_tensor = q_ids.repeat(len(chunk_doc_ids), 1)
                logits = model(q_tensor, doc_tensor)
                probs = torch.sigmoid(logits).cpu().numpy().tolist()
                scores.extend(probs)
            doc_score_pairs = list(zip(target_doc_ids, scores))
            doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
            rankings[qid] = doc_score_pairs
    metrics = calculate_metrics(rankings, qrels, k_list=[5, 10])
    return metrics, rankings

print('=' * 75)
print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Val Loss':<10} | {'Val MRR':<10} | {'Val MAP':<10} | {'Val NDCG@5':<10}")
print('=' * 75)

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        q_ids = batch['query_ids'].to(device)
        d_ids = batch['doc_ids'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(q_ids, d_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * len(labels)
        
    avg_train_loss = total_train_loss / len(train_dataset)
    
    val_metrics, _ = evaluate_model(val_qids, candidate_pool_size=150)
    val_map = val_metrics.get('MAP', 0.0)
    val_mrr = val_metrics.get('MRR', 0.0)
    val_ndcg5 = val_metrics.get('nDCG@5', 0.0)
    
    # Val BCE loss
    model.eval()
    val_bce_loss, val_count = 0.0, 0
    with torch.no_grad():
        for qid in val_qids:
            pos_docs = qrels.get(qid, set())
            non_rel = list(set(casedocs.keys()) - pos_docs)[:10]
            for d_id in list(pos_docs) + non_rel:
                if d_id in casedocs:
                    q_s = pad_sequence(vocab.text_to_ids(queries[qid], max_len=CONFIG['MAX_QUERY_LENGTH']), CONFIG['MAX_QUERY_LENGTH'])
                    d_s = pad_sequence(vocab.text_to_ids(casedocs[d_id], max_len=CONFIG['MAX_DOCUMENT_LENGTH']), CONFIG['MAX_DOCUMENT_LENGTH'])
                    q_t = torch.tensor([q_s], dtype=torch.long, device=device)
                    d_t = torch.tensor([d_s], dtype=torch.long, device=device)
                    target = torch.tensor([1.0 if d_id in pos_docs else 0.0], dtype=torch.float32, device=device)
                    val_bce_loss += criterion(model(q_t, d_t), target).item()
                    val_count += 1
    avg_val_loss = val_bce_loss / max(val_count, 1)
    
    history.append({
        'epoch': epoch,
        'train_loss': round(avg_train_loss, 4),
        'val_loss': round(avg_val_loss, 4),
        'val_mrr': round(val_mrr, 4),
        'val_map': round(val_map, 4),
        'val_ndcg5': round(val_ndcg5, 4)
    })
    
    print(f"{epoch:<6} | {avg_train_loss:<12.4f} | {avg_val_loss:<10.4f} | {val_mrr:<10.4f} | {val_map:<10.4f} | {val_ndcg5:<10.4f}")
    
    if val_map > best_val_map:
        best_val_map = val_map
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab': vocab.w2i,
            'config': CONFIG
        }, checkpoint_path)
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['EARLY_STOPPING_PATIENCE']:
            print(f'\n[Early Stopping] Triggered at epoch {epoch}. Best Val MAP: {best_val_map:.4f} at epoch {best_epoch}.')
            break

print('=' * 75)
print(f'Training complete. Best checkpoint saved to {checkpoint_path}')

In [ ]:
# Cell 9: Test Set Evaluation & Artifact Saving
from src.evaluation import save_metrics_csv, save_predictions_csv, save_history_csv, save_loss_plot

results_dir = os.path.join(project_root, 'results', 'bilstm')
os.makedirs(results_dir, exist_ok=True)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_metrics, test_rankings = evaluate_model(test_qids, candidate_pool_size=None)

# Save required outputs
save_metrics_csv(test_metrics, model_name='BiLSTM', save_path=os.path.join(results_dir, 'metrics.csv'))
save_predictions_csv(test_rankings, qrels, save_path=os.path.join(results_dir, 'predictions.csv'))
save_history_csv(history, save_path=os.path.join(results_dir, 'training_history.csv'))
save_loss_plot(history, save_path=os.path.join(results_dir, 'loss_plot.png'))

# Display summary dataframe
metrics_df = pd.read_csv(os.path.join(results_dir, 'metrics.csv'))
print('\nFinal Research Output:')
metrics_df

In [ ]:
# Cell 10: Loss Curves Plot Visualization
from PIL import Image
loss_plot_path = os.path.join(project_root, 'results', 'bilstm', 'loss_plot.png')
if os.path.exists(loss_plot_path):
    img = Image.open(loss_plot_path)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()